# 01 Explore Detections
Qualitative inspection of YOLOX-S on COCO val2017.

In [ ]:
import sys
sys.path.insert(0, "..")

import os
import json
import random
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from pycocotools.coco import COCO

from src.detector import DetectorWrapper, Detection
from src.utils import load_image

DATA_DIR = Path("../data/coco")
VAL_DIR = DATA_DIR / "val2017"
ANN_FILE = DATA_DIR / "annotations" / "instances_val2017.json"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

if not VAL_DIR.exists():
    print(
        f"ERROR: COCO val2017 directory not found at {VAL_DIR.resolve()}.\n"
        "Run notebooks/00_setup_and_data.ipynb first, or verify that this "
        "notebook is being executed from the notebooks/ directory."
    )
if not ANN_FILE.exists():
    print(
        f"ERROR: COCO annotation file not found at {ANN_FILE.resolve()}.\n"
        "Expected ../data/coco/annotations/instances_val2017.json."
    )

assert DATA_DIR.exists(), f"Missing COCO data directory: {DATA_DIR.resolve()}"
assert VAL_DIR.exists(), f"Missing COCO val2017 images: {VAL_DIR.resolve()}"
assert ANN_FILE.exists(), f"Missing COCO val2017 annotations: {ANN_FILE.resolve()}"

print(f"Working directory : {Path.cwd().resolve()}")
print(f"COCO data dir     : {DATA_DIR.resolve()}")
print(f"val2017 images    : {VAL_DIR.resolve()}")
print(f"annotations       : {ANN_FILE.resolve()}")

## 1. Load COCO Annotations

In [ ]:
coco_val = COCO(str(ANN_FILE))

img_ids = coco_val.getImgIds()
cat_ids = coco_val.getCatIds()
ann_ids = coco_val.getAnnIds()
anns = coco_val.loadAnns(ann_ids)

cat_id_to_name = {cat["id"]: cat["name"] for cat in coco_val.loadCats(cat_ids)}
category_counter = Counter(cat_id_to_name[ann["category_id"]] for ann in anns)

print(f"Total images      : {len(img_ids):,}")
print(f"Total categories  : {len(cat_ids):,}")
print(f"Total annotations : {len(ann_ids):,}")

print("\n10 most common categories by annotation count:")
for name, count in category_counter.most_common(10):
    print(f"  {name:<16} {count:>6,}")

print("\n10 least common categories by annotation count:")
for name, count in sorted(category_counter.items(), key=lambda item: item[1])[:10]:
    print(f"  {name:<16} {count:>6,}")

## 2. Load YOLOX-S Detector

In [ ]:
import torch

detector = DetectorWrapper(model_name="yolox-s", config_path="../config/detector_config.yaml")
detector.load_model()

print(f"Model name       : {detector.model_name}")
print(f"Device           : {detector.device}")
if torch.cuda.is_available():
    print(f"GPU              : {torch.cuda.get_device_name(0)}")
else:
    print("GPU              : unavailable; running on CPU")
print(f"Default conf_thr : {detector.default_conf_thresh:.2f}")
print(f"Default nms_thr  : {detector.default_nms_thresh:.2f}")

## 3. Single Image: Full Inspection
Detailed look at one COCO image.

In [ ]:
target_image_id = 139
img_info = coco_val.loadImgs(target_image_id)[0]
image_path = VAL_DIR / img_info["file_name"]
assert image_path.exists(), f"Missing image file: {image_path.resolve()}"

image = load_image(str(image_path))
detections = detector.detect(image)

height, width = image.shape[:2]
print(f"Image ID         : {target_image_id}")
print(f"Filename         : {img_info['file_name']}")
print(f"Image size       : {width} x {height}")
print(f"Num detections   : {len(detections)}")

for det in detections:
    print(
        f"  #{det.detection_id:02d} {det.class_name:<14} "
        f"conf={det.confidence:.3f} bbox={det.bbox} "
        f"area={det.area} size={det.relative_size}"
    )

fig, ax = plt.subplots(1, 1, figsize=(12, 9))
ax.imshow(image)
ax.set_title(f"YOLOX-S: {len(detections)} detections")
ax.axis("off")

for det in detections:
    x1, y1, x2, y2 = det.bbox
    rect = patches.Rectangle(
        (x1, y1), x2 - x1, y2 - y1,
        linewidth=2, edgecolor="lime", facecolor="none"
    )
    ax.add_patch(rect)
    ax.text(
        x1, max(y1 - 4, 0), f"{det.class_name} {det.confidence:.2f}",
        color="lime", fontsize=8, fontweight="bold",
        bbox=dict(facecolor="black", alpha=0.5, pad=1, edgecolor="none")
    )

plt.tight_layout()
plt.show()

## 4. Threshold Sensitivity
How conf_thresh and nms_thresh affect detection count.

In [ ]:
conf_values = [0.1, 0.25, 0.4, 0.6, 0.8]
conf_counts = []
for value in conf_values:
    dets = detector.detect(image, conf_thresh=value, nms_thresh=0.45)
    conf_counts.append(len(dets))

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.plot(conf_values, conf_counts, marker="o", linewidth=2)
ax.set_xlabel("conf_thresh")
ax.set_ylabel("num_detections")
ax.set_title("Detection count vs conf_thresh")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

nms_values = [0.3, 0.4, 0.5, 0.6, 0.7]
nms_counts = []
for value in nms_values:
    dets = detector.detect(image, conf_thresh=0.25, nms_thresh=value)
    nms_counts.append(len(dets))

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.plot(nms_values, nms_counts, marker="o", linewidth=2, color="tab:orange")
ax.set_xlabel("nms_thresh")
ax.set_ylabel("num_detections")
ax.set_title("Detection count vs nms_thresh")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Confidence threshold sweep:")
for value, count in zip(conf_values, conf_counts):
    print(f"  conf_thresh={value:.2f}: {count:3d} detections")

print("\nNMS threshold sweep:")
for value, count in zip(nms_values, nms_counts):
    print(f"  nms_thresh={value:.2f}: {count:3d} detections")

Higher `conf_thresh` reduces false positives but may miss true detections. Lower NMS threshold reduces duplicate boxes in crowded scenes.

## 5. Detection Distribution Across 50 Images

In [ ]:
sample_image_ids_50 = random.sample(img_ids, k=min(50, len(img_ids)))
sample_results_50 = []
class_counter_50 = Counter()
detection_counts_per_image = []

for index, image_id in enumerate(sample_image_ids_50, start=1):
    info = coco_val.loadImgs(image_id)[0]
    path = VAL_DIR / info["file_name"]
    img = load_image(str(path))
    dets = detector.detect(img, conf_thresh=0.25, nms_thresh=0.45)
    sample_results_50.append({"image_id": image_id, "filename": info["file_name"], "detections": dets})
    class_counter_50.update(det.class_name for det in dets)
    detection_counts_per_image.append(len(dets))
    if index % 10 == 0 or index == len(sample_image_ids_50):
        print(f"Processed {index:2d}/{len(sample_image_ids_50)} images")

top_classes = class_counter_50.most_common(15)
labels = [name for name, _ in top_classes][::-1]
counts = [count for _, count in top_classes][::-1]

fig, ax = plt.subplots(1, 1, figsize=(8, 6))
ax.barh(labels, counts, color="tab:blue")
ax.set_xlabel("detection count")
ax.set_title("Most Detected Classes (50 images, YOLOX-S)")
plt.tight_layout()
plt.show()

counts_array = np.asarray(detection_counts_per_image, dtype=np.float32)
print("Detection count summary across 50 images:")
print(f"  mean: {counts_array.mean():.2f}")
print(f"  std : {counts_array.std():.2f}")
print(f"  min : {int(counts_array.min())}")
print(f"  max : {int(counts_array.max())}")

## 6. Detection Grid (12 images)

In [ ]:
grid_image_ids = random.sample(img_ids, k=min(12, len(img_ids)))
cmap = plt.cm.get_cmap("tab20", 20)

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for ax, image_id in zip(axes, grid_image_ids):
    info = coco_val.loadImgs(image_id)[0]
    path = VAL_DIR / info["file_name"]
    img = load_image(str(path))
    dets = detector.detect(img, conf_thresh=0.25, nms_thresh=0.45)

    ax.imshow(img)
    ax.set_title(f"{info['file_name']} ({len(dets)} dets)", fontsize=9)
    ax.axis("off")

    for det in dets:
        x1, y1, x2, y2 = det.bbox
        color = cmap(det.class_id % 20)
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=1.5, edgecolor=color, facecolor="none"
        )
        ax.add_patch(rect)

for ax in axes[len(grid_image_ids):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

## 7. Relative Size Distribution
Small/medium/large breakdown across 50 images.

In [ ]:
if "sample_results_50" not in globals():
    print("Section 5 results not found; recomputing 50-image detection sample.")
    sample_image_ids_50 = random.sample(img_ids, k=min(50, len(img_ids)))
    sample_results_50 = []
    for image_id in sample_image_ids_50:
        info = coco_val.loadImgs(image_id)[0]
        img = load_image(str(VAL_DIR / info["file_name"]))
        dets = detector.detect(img, conf_thresh=0.25, nms_thresh=0.45)
        sample_results_50.append({"image_id": image_id, "filename": info["file_name"], "detections": dets})

size_counter = Counter()
for result in sample_results_50:
    size_counter.update(det.relative_size for det in result["detections"])

size_labels = ["small", "medium", "large"]
size_counts = [size_counter.get(label, 0) for label in size_labels]

fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.pie(size_counts, labels=size_labels, autopct="%1.1f%%", startangle=90)
ax.set_title("Detection Size Distribution")
plt.tight_layout()
plt.show()

print("Relative size counts:")
for label, count in zip(size_labels, size_counts):
    print(f"  {label:<6}: {count}")

print(
    "\nSmall objects are harder for GradCAM because gradients can become diffuse. "
    "The XAI selector uses relative_size as a key feature."
)

## 8. Cleanup

In [ ]:
detector.unload_model()
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Detector unloaded. GPU memory released.")